# Tutorial 08: Comparing Propagation Formalisms

This notebook compares the three propagation formalisms on a miniature version of the magnetic Fourier-transform-holography experiment used in the other tutorials. The sample contains a magnetic multilayer, one object hole (OH), and one reference hole (RH). For each formalism we propagate right- and left-circular illumination, then compare:

1. complex real-space exit waves;
2. detector-plane holograms;
3. FTH reconstructions of the helicity difference `CR - CL`.

The geometry is intentionally modest so the full notebook remains interactive. Scalar propagation is an eigenmode approximation. Jones carries `[Ex, Ey]`. Mueller–Stokes publishes `[I, Q, U, V]` while retaining the coherent carrier modes required for diffraction. The baseline runs use pure `CR`/`CL`; the final Stokes-only section lets you switch on partially polarized incident beams.

The sample-to-detector selector below defaults to the fast Fraunhofer FFT. See [Tutorial 16](16_compare_detector_propagation.ipynb) for the same-exit-wave Rayleigh–Sommerfeld comparison. 


In [ ]:
# Sample-to-detector propagation (independent of multislice).
detector_propagation_method = "fraunhofer"  # Default; opt in with "rayleigh_sommerfeld".
# Direct Rayleigh-Sommerfeld is expensive: try small grids first.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from scattering_calculator.beam_propagator import Stokes_propagator
from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim

# When re-running this notebook in an already-open kernel, force the local
# modules to refresh.  Otherwise Python may keep an older in-memory
# simulation_configuration module that does not yet pass `input_stokes` to
# the Stokes propagator, making the final CL−CR sweep collapse to exact zero.
Stokes_propagator = importlib.reload(Stokes_propagator)
sim = importlib.reload(sim)
validate_stokes = Stokes_propagator.validate_stokes

METHODS = ("Scalar", "Jones", "Stokes")
# The baseline comparison uses pure, fully polarized helicity eigenstates.
POLARIZATIONS = ("CR", "CL")

## 1. Experimental geometry and magnetic material

The material recipe is the same style used by the CK tutorials: an opaque top stack, a SiN membrane, and a Pt/Co/Pt magnetic layer. The detector determines the real-space sampling. Detector noise and a beamstop are omitted here so differences between propagation formalisms remain visible.

In [ ]:
detector_shape = (64, 64)
oversampling = 2
detector_center = tuple(np.array(detector_shape) // 2)

xray = sim.XRayConfig(
    energy=778.0, photon_flux=5e8, pol="CR", coherence_length=(10e-6, 10e-6)
)
xray.setup()

detector = sim.DetectorConfig(
    detector_propagation_method=detector_propagation_method,
    shape=detector_shape,
    pixel_size=20e-6,
    sample_to_detector_distance=0.075,
    detector_center=detector_center,
    detector_params={
        "readout_noise_average": 0.0,
        "readout_noise_sigma": 0.0,
        "counts_per_photon": 1.0,
        "quantum_efficiency": 1.0,
        "noise_seed": 7,
    },
    measurement_config={"exposure_time": 1.0, "number_frames": 1},
)
detector.setup()
real_space_pixel_size = detector.calc_realspace_resolution(xray.beam_params) / oversampling
# SampleConfig fills the layer count in place, so this must remain a list.
sample_shape = [0, oversampling * detector_shape[0], oversampling * detector_shape[1]]

sample = sim.SampleConfig(
    recipe="Au(1000)/SiN(80)/Pt(4)Co(18)Pt(2)",
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xray,
    sample_name="three-formalism magnetic FTH comparison",
)
sample.setup()
print(f"sample grid: {sample_shape[1:]}")
print(f"real-space pixel: {real_space_pixel_size * 1e9:.2f} nm")
print("layers:", sample.sample_structure.layer_names)

## 2. Magnetic domains and FTH apertures

A smooth labyrinth-like out-of-plane magnetization fills the magnetic layer. The front aperture contains one large object hole over those domains and one small, displaced reference hole. The reference hole creates the shifted real-space copy in the FTH reconstruction.

In [ ]:
ny, nx = sample_shape[1:]
y, x = np.mgrid[-1:1:ny*1j, -1:1:nx*1j]
domain_phase = 1.0 * np.sin(60.0 * y) #+ 7.0 * np.sin(12.0 * x )#+ 1.8 * np.sin(7.0 * x * 7.0 * y -70)
mz = np.tanh(np.sin(domain_phase) / 0.22)
magnetization = pattern_generator.map_magnetization_to_3d(
    magnetic_pattern_x=np.zeros_like(mz),
    magnetic_pattern_y=np.sqrt(np.clip(1.0 - mz**2, 0.0, 1.0)),
    magnetic_pattern_z=mz,
    nr_repeats=sample.sample_structure.sample_shape[0],
)
sample.assign_magnetic_pattern(magnetization)

thicknesses = sample.sample_structure.layer_thicknesses
membrane_index = sample.sample_structure.layer_names.index("SiN")
aperture = sim.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=thicknesses,
    aperture_layer_names=sample.sample_structure.layer_names,
    aperture_config={
        "apertures_type": ["OH", "RH"],
        "apertures_radius": [480e-9, 45e-9],
        "apertures_center": [(0.0, 0.0), (1050e-9, -980e-9)],
        "apertures_sigma": [3e-9, 1e-9],
        "apertures_angle": [0.0, 0.0],
        "apertures_ellipticity": [1.0, 1.0],
        "apertures_roughness": [0.0, 0.0],
        "apertures_roughness_modes": [(0, 0), (0, 0)],
        "apertures_seed": [1, 2],
        "apertures_top_radius_factor": [1.0, 1.0],
        "aperture_taper_depth": 0.0,
        "thickness_OH": float(np.sum(thicknesses[:membrane_index])),
    },
    use_roi=True,
)
aperture.setup()
aperture_mask = aperture.return_aperture()
sample.assign_aperture_mask(aperture_mask)

projection = np.max(aperture_mask, axis=0)
extent_um = np.array([-nx/2, nx/2, ny/2, -ny/2]) * real_space_pixel_size * 1e6
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
axes[0].imshow(mz, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um)
axes[0].set_title("magnetic $m_z$")
axes[1].imshow(projection, cmap="gray", vmin=0, vmax=1, extent=extent_um)
axes[1].set_title("OH + RH aperture")
axes[2].imshow(mz * projection, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um)
axes[2].set_title("domains visible through mask")
for ax in axes:
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
plt.show()

## 3. Build the material response and illumination

Jones and Stokes consume the same dielectric tensor. Scalar constructs its polarization-specific refractive-index stack when each helicity is propagated. A broad Gaussian illuminates both holes.

In [ ]:
sample.sample_structure.calculate_final_dielectric_tensor(
    use_aperture_roi=True, compact=True
)
illumination = sim.IlluminationConfig(
    XRayConfig=xray,
    shape=sample_shape[1:],
    real_space_pixel_size=real_space_pixel_size,
    illumination_function="gaussian",
    illumination_config={
        "center": np.array([0.0, 0.0]),
        "distance": 0.0,
        "fwhm": 3.5e-6,
        "alpha_beam": (0.0, 0.0),
    },
)
illumination.setup()

## 4. Propagate CR and CL with every formalism

The material interaction is evaluated layer by layer without transverse multislice diffraction (`propagate=False`), matching the thin-object FTH approximation. Fraunhofer propagation to the detector is always performed. These baseline runs use pure `CR` and `CL` illumination. The complex carrier retained by Stokes is used only where a phase-bearing exit wave is required.

In [ ]:
results = {method: {} for method in METHODS}

for method in METHODS:
    for pol in POLARIZATIONS:
        illumination.update_polarization(pol)
        propagation = sim.SamplePropagatorConfig(
            SampleConfig=sample,
            IlluminationConfig=illumination,
            propagator_method=method,
            propagator_config={
                "propagate": False,
                "jones_apply_zero_order_phase": True,
                "scalar_apply_zero_order_phase": True,
                "dielectric_tensor_use_roi": True,
                "scalar_refractive_index_lazy": True,
            },
        )
        propagation.setup()
        wavefront = propagation.return_wavefront()
        detector.assign_propagated_wavefront(propagation)
        detector.detect_hologram()
        results[method][pol] = {
            "exit": propagation.return_scalar_wavefield().copy(),
            # Selected propagation model on physical detector pixels; no noise.
            "hologram": detector.return_ideal_hologram().copy(),
            "stokes": getattr(wavefront, "exit_stokes", None),
        }

print("Completed:", [(m, p) for m in METHODS for p in POLARIZATIONS])

## 5. Real-space exit waves

Amplitude and phase are shown for the CR exit wave. The object and reference holes are visible directly; their relative phase produces the holographic interference pattern in reciprocal space.

In [ ]:
fig, axes = plt.subplots(len(METHODS), 2, figsize=(8, 9), constrained_layout=True)
for row, method in enumerate(METHODS):
    exit_wave = results[method]["CR"]["exit"]
    amp = axes[row, 0].imshow(np.abs(exit_wave), cmap="magma", extent=extent_um)
    phase = axes[row, 1].imshow(
        np.angle(exit_wave), cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent_um
    )
    axes[row, 0].set_title(f"{method}: |exit wave|")
    axes[row, 1].set_title(f"{method}: exit-wave phase")
    fig.colorbar(amp, ax=axes[row, 0], shrink=0.78)
    fig.colorbar(phase, ax=axes[row, 1], shrink=0.78)
for ax in axes.flat:
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
plt.show()

## 6. Detector holograms

The first row shows the CR hologram on a logarithmic scale. The second row shows the magnetic helicity difference `CR - CL` on a symmetric linear scale.

In [ ]:
fig, axes = plt.subplots(2, len(METHODS), figsize=(12, 7), constrained_layout=True)
for col, method in enumerate(METHODS):
    cr = results[method]["CR"]["hologram"]
    cl = results[method]["CL"]["hologram"]
    floor = max(cr.max() * 1e-8, np.finfo(float).tiny)
    top = axes[0, col].imshow(np.log10(np.maximum(cr, floor)), cmap="magma")
    axes[0, col].set_title(f"{method}: log10 CR hologram")
    fig.colorbar(top, ax=axes[0, col], shrink=0.78)

    difference = cr - cl
    limit = max(np.max(np.abs(difference)), np.finfo(float).eps)
    bottom = axes[1, col].imshow(difference, cmap="RdBu_r", vmin=-limit, vmax=limit)
    axes[1, col].set_title(f"{method}: CR − CL")
    fig.colorbar(bottom, ax=axes[1, col], shrink=0.78)
for ax in axes.flat:
    ax.set_axis_off()
plt.show()

## 7. FTH reconstructions

FTH reconstruction is the centered two-dimensional Fourier transform of the hologram. Here it is applied to the helicity difference to emphasize magnetic contrast. The displaced object replicas occur at the OH–RH separation; the central autocorrelation is also visible.

In [ ]:
def fth_reconstruct(hologram):
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(hologram)))

reconstructions = {}
fig, axes = plt.subplots(1, len(METHODS), figsize=(12, 3.8), constrained_layout=True)
for ax, method in zip(axes, METHODS):
    difference = results[method]["CR"]["hologram"] - results[method]["CL"]["hologram"]
    reconstruction = fth_reconstruct(difference)
    reconstructions[method] = reconstruction
    magnitude = np.real(reconstruction)
    image = ax.imshow(magnitude, cmap="inferno", vmax=np.percentile(magnitude, 93.7))
    ax.set_title(f"{method}: |FTH(CR − CL)|")
    ax.set_axis_off()
    fig.colorbar(image, ax=ax, shrink=0.78)
plt.show()

## 8. Quantitative comparison and Stokes output

Jones is used as the reference. For the deterministic dielectric tensor, Stokes detector intensity should agree with Jones to floating-point precision. Scalar may differ where polarization mixing matters. The final panel exposes the four Stokes components at the sample exit.

In [ ]:
def relative_l2(candidate, reference):
    denominator = np.linalg.norm(reference)
    return np.linalg.norm(candidate - reference) / denominator if denominator else np.nan

for method in METHODS:
    hologram_error = relative_l2(
        results[method]["CR"]["hologram"], results["Jones"]["CR"]["hologram"]
    )
    reconstruction_error = relative_l2(reconstructions[method], reconstructions["Jones"])
    print(f"{method:7s}  CR hologram error={hologram_error:.3e}  reconstruction error={reconstruction_error:.3e}")

stokes_exit = results["Stokes"]["CR"]["stokes"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.3), constrained_layout=True)
for index, (ax, label) in enumerate(zip(axes, ["I", "Q", "U", "V"])):
    component = stokes_exit[..., index]
    if index == 0:
        image = ax.imshow(component, cmap="magma", extent=extent_um)
    else:
        limit = max(np.max(np.abs(component)), np.finfo(float).eps)
        image = ax.imshow(component, cmap="RdBu_r", vmin=-limit, vmax=limit, extent=extent_um)
    ax.set_title(f"Stokes {label}")
    ax.set_xlabel("x (µm)")
    fig.colorbar(image, ax=ax, shrink=0.75)
plt.show()

## 9. Stokes-only sweep: decreasing circular polarization

The comparison above intentionally used only pure `CR` and `CL` helicities so Scalar, Jones, and Stokes were compared on the same footing. Now we switch to Stokes only and decrease the degree of circular polarization. For every degree `p`, we propagate a matched helicity pair: `CR = [1, 0, 0, +p]` and `CL = [1, 0, 0, -p]`. Detector magnetic contrast and FTH reconstructions are then computed as `CL − CR` at the same `p`.

In [ ]:
# Stokes vectors are ordered [I, Q, U, V].  With this project convention,
# pure CR is [1, 0, 0, +1] and pure CL is [1, 0, 0, -1].  Values with
# |V| < I represent a circularly polarized fraction plus an unpolarized
# fraction.  For example, p=0.7 is 70% circular + 30% unpolarized.
POLARIZATION_DEGREES = {
    "100%": 1.0,
    "70%": 0.7,
    "30%": 0.3,
    "0%": 0.0,
}
PARTIAL_STOKES_CASES = {
    degree_label: {
        "CR": np.array([1.0, 0.0, 0.0, +degree]),
        "CL": np.array([1.0, 0.0, 0.0, -degree]),
    }
    for degree_label, degree in POLARIZATION_DEGREES.items()
}
for helicity_pair in PARTIAL_STOKES_CASES.values():
    for stokes_vector in helicity_pair.values():
        validate_stokes(stokes_vector)

def average_normalized_stokes(stokes_field):
    """Return the spatially averaged Stokes state normalized by total I."""
    total_intensity = np.sum(stokes_field[..., 0])
    if total_intensity <= 0:
        raise ValueError("Cannot normalize a Stokes field with zero total intensity.")
    return np.sum(stokes_field, axis=(0, 1)) / total_intensity

partial_results = {degree_label: {} for degree_label in PARTIAL_STOKES_CASES}
# The polarization label here is only used to build the Gaussian spatial envelope.
# The physical incident polarization is replaced by `input_stokes` below.
illumination.update_polarization("CR")

for degree_label, helicity_pair in PARTIAL_STOKES_CASES.items():
    for helicity, input_stokes in helicity_pair.items():
        propagation = sim.SamplePropagatorConfig(
            SampleConfig=sample,
            IlluminationConfig=illumination,
            propagator_method="Stokes",
            propagator_config={
                "propagate": False,
                "jones_apply_zero_order_phase": True,
                "dielectric_tensor_use_roi": True,
                # This is the key Stokes knob: one uniform Stokes vector [I, Q, U, V].
                # Values with sqrt(Q^2 + U^2 + V^2) < I are partially polarized.
                "input_stokes": input_stokes,
            },
        )
        propagation.setup()
        wavefront = propagation.return_wavefront()
        detector.assign_propagated_wavefront(propagation)
        detector.detect_hologram()
        actual_input_state = average_normalized_stokes(wavefront.input_stokes)
        expected_input_state = input_stokes / input_stokes[0]
        np.testing.assert_allclose(
            actual_input_state,
            expected_input_state,
            atol=1e-10,
            err_msg=(
                "The Stokes propagator did not receive the requested input_stokes. "
                "Restart the kernel or rerun the imports/reload cell."
            ),
        )
        partial_results[degree_label][helicity] = {
            "input_stokes": input_stokes,
            "actual_input_state": actual_input_state,
            "exit": propagation.return_scalar_wavefield().copy(),
            "stokes": wavefront.exit_stokes.copy(),
            "hologram": detector.return_ideal_hologram().copy(),
        }

for degree_label, helicity_pair in partial_results.items():
    cr = helicity_pair["CR"]["input_stokes"]
    cl = helicity_pair["CL"]["input_stokes"]
    actual_cr = helicity_pair["CR"]["actual_input_state"]
    actual_cl = helicity_pair["CL"]["actual_input_state"]
    degree = np.linalg.norm(cr[1:]) / cr[0]
    print(
        f"{degree_label:>4s}: requested CR V={cr[3]:+.2f}, CL V={cl[3]:+.2f}; "
        f"actual CR V={actual_cr[3]:+.2f}, CL V={actual_cl[3]:+.2f}; "
        f"matched degree={degree:.2f}"
    )

The first row shows the `CR` exit-plane intensity `I`. The second row shows the matched exit-plane helicity difference `CL(I) − CR(I)`. As the degree of circular polarization decreases, the helicity-dependent exit contrast should decrease toward zero.

In [ ]:
degree_labels = list(partial_results)
fig, axes = plt.subplots(2, len(degree_labels), figsize=(4 * len(degree_labels), 6), constrained_layout=True)
if len(degree_labels) == 1:
    axes = axes[:, np.newaxis]

for col, degree_label in enumerate(degree_labels):
    cr_exit = partial_results[degree_label]["CR"]["stokes"]
    cl_exit = partial_results[degree_label]["CL"]["stokes"]
    image_i = axes[0, col].imshow(cr_exit[..., 0], cmap="magma", extent=extent_um)
    axes[0, col].set_title(f"{degree_label}: CR exit I")
    fig.colorbar(image_i, ax=axes[0, col], shrink=0.78)

    exit_difference = cl_exit[..., 0] - cr_exit[..., 0]
    limit = max(np.max(np.abs(exit_difference)), np.finfo(float).eps)
    image_diff = axes[1, col].imshow(
        exit_difference, cmap="RdBu_r", vmin=-limit, vmax=limit, extent=extent_um
    )
    axes[1, col].set_title(f"{degree_label}: exit CL − CR")
    fig.colorbar(image_diff, ax=axes[1, col], shrink=0.78)

for ax in axes.flat:
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
plt.show()

Finally, compare detector holograms and FTH reconstructions for the same polarization sweep. For each degree, the hologram difference is computed from the matched pair `CL − CR`, and the FTH reconstruction is the centered Fourier transform of that matched helicity difference.

In [ ]:
partial_differences = {}
partial_reconstructions = {}
for degree_label in degree_labels:
    cr_hologram = partial_results[degree_label]["CR"]["hologram"]
    cl_hologram = partial_results[degree_label]["CL"]["hologram"]
    hologram_difference = cl_hologram - cr_hologram
    reconstruction = fth_reconstruct(hologram_difference)
    partial_differences[degree_label] = hologram_difference
    partial_reconstructions[degree_label] = reconstruction

reference_norm = max(
    np.linalg.norm(partial_differences[degree_labels[0]]), np.finfo(float).eps
)
for degree_label in degree_labels:
    diff_norm = np.linalg.norm(partial_differences[degree_label])
    recon_norm = np.linalg.norm(partial_reconstructions[degree_label])
    print(
        f"{degree_label:>4s}: ||CL−CR hologram|| / 100% = "
        f"{diff_norm / reference_norm:.3f}; ||FTH||={recon_norm:.3e}"
    )

max_hologram_diff = max(
    np.max(np.abs(diff)) for diff in partial_differences.values()
)
max_hologram_diff = max(max_hologram_diff, np.finfo(float).eps)
max_reconstruction = max(
    0.015*np.max(np.abs(reconstruction)) for reconstruction in partial_reconstructions.values()
)
max_reconstruction = max(max_reconstruction, np.finfo(float).eps)

fig, axes = plt.subplots(3, len(degree_labels), figsize=(4 * len(degree_labels), 9), constrained_layout=True)
if len(degree_labels) == 1:
    axes = axes[:, np.newaxis]

for col, degree_label in enumerate(degree_labels):
    cr_hologram = partial_results[degree_label]["CR"]["hologram"]
    hologram_difference = partial_differences[degree_label]
    reconstruction = partial_reconstructions[degree_label]

    floor = max(cr_hologram.max() * 1e-8, np.finfo(float).tiny)
    image_h = axes[0, col].imshow(np.log10(np.maximum(cr_hologram, floor)), cmap="magma")
    axes[0, col].set_title(f"{degree_label}: log10 CR hologram")
    axes[0, col].set_axis_off()
    fig.colorbar(image_h, ax=axes[0, col], shrink=0.78)

    image_d = axes[1, col].imshow(
        hologram_difference,
        cmap="RdBu_r",
        vmin=-max_hologram_diff,
        vmax=max_hologram_diff,
    )
    axes[1, col].set_title(f"{degree_label}: hologram CL − CR")
    axes[1, col].set_axis_off()
    fig.colorbar(image_d, ax=axes[1, col], shrink=0.78)

    image_r = axes[2, col].imshow(
        np.abs(reconstruction), cmap="inferno", vmin=0.0, vmax=max_reconstruction
    )
    axes[2, col].set_title(f"{degree_label}: |FTH(CL − CR)|")
    axes[2, col].set_axis_off()
    fig.colorbar(image_r, ax=axes[2, col], shrink=0.78)
plt.show()

## Interpretation

- The **exit-wave plots** show the object and reference transmissions before detector propagation.
- The **detector holograms** contain interference between the OH and RH waves. Their helicity difference isolates magnetic circular contrast.
- The **FTH reconstructions** turn that interference into displaced real-space images of the magnetic object.
- **Jones and Stokes** describe the same deterministic, non-depolarizing material response for pure inputs, so their intensities coincide. Stokes additionally exposes `I, Q, U, V` and can accept partially polarized `input_stokes` beams.
- In the final Stokes-only sweep, every FTH reconstruction uses a matched helicity difference, `CL − CR`, at the same degree of polarization.
- **Scalar** is fastest, but assumes the selected polarization remains an eigenmode and can depart from the vector descriptions when the sample mixes polarization.